# 02 — General Overview

Landscape of kidney stone (ICD-10: N20) hospitalizations in São Paulo, with statistical rigor.

**Scope:** 206,500 admissions × 510 hospitals × 10 years (2016–2025)

**Report:** `reports/02_general_overview.md`

**Sections:** Scale & trend, demographics, geography, procedure mix, hospital classification, LOS distribution, admission type, concentration analysis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from shared import (
    load_kidney, load_hospital_tags, setup_plot_style,
    save_plot, save_metrics, PLOT_DIR, CATEGORY_MAP, PROC_NAMES,
)

setup_plot_style()
kidney = load_kidney()
tags = load_hospital_tags()
print(f"Loaded {len(kidney):,} admissions, {len(tags)} hospitals")
print(f"Years: {int(kidney['year'].min())}–{int(kidney['year'].max())}")
print(f"Sub-diagnoses: {kidney['DIAG_PRINC'].nunique()}")

Loaded 206,500 admissions, 510 hospitals
Years: 2015–2025
Sub-diagnoses: 5


## 1. Scale & Yearly Trend

In [2]:
yearly = kidney[kidney["year"] >= 2016].groupby("year").agg(
    admissions=pd.NamedAgg(column="year", aggfunc="count"),
    mean_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="mean"),
    median_los=pd.NamedAgg(column="DIAS_PERM", aggfunc="median"),
    total_cost=pd.NamedAgg(column="VAL_TOT", aggfunc="sum"),
    deaths=pd.NamedAgg(column="MORTE", aggfunc="sum"),
    pct_emergency=pd.NamedAgg(column="is_emergency", aggfunc="mean"),
).reset_index()
yearly["mort_rate"] = yearly["deaths"] / yearly["admissions"] * 100
yearly["yoy_growth"] = yearly["admissions"].pct_change() * 100

# Trend test (Kendall tau for monotonic trend)
tau, p = stats.kendalltau(range(len(yearly)), yearly["admissions"])

# CAGR (2016–2025, full period)
n_yrs = int(yearly.iloc[-1]["year"] - yearly.iloc[0]["year"])
cagr = (yearly.iloc[-1]["admissions"] / yearly.iloc[0]["admissions"]) ** (1 / n_yrs) - 1

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

bars = axes[0].bar(yearly["year"], yearly["admissions"], color="#2563EB")
axes[0].set_title(f"Admissions per Year  (CAGR: {cagr*100:.1f}%)")
axes[0].set_ylabel("Count")
for i, row in yearly.iterrows():
    if i > 0:
        axes[0].annotate(f"{row['yoy_growth']:+.0f}%",
                         xy=(row["year"], row["admissions"]),
                         ha="center", va="bottom", fontsize=7, color="#6B7280")

axes[1].plot(yearly["year"], yearly["mean_los"], "o-", color="#D97706", label="Mean")
axes[1].plot(yearly["year"], yearly["median_los"], "s--", color="#DC2626", label="Median")
axes[1].set_title("Length of Stay Trend")
axes[1].set_ylabel("Days")
axes[1].legend()

axes[2].bar(yearly["year"], yearly["total_cost"] / 1e6, color="#059669")
axes[2].set_title("Total Cost per Year")
axes[2].set_ylabel("BRL (millions)")

for ax in axes:
    ax.set_xlabel("Year")
    ax.tick_params(axis="x", rotation=45)

fig.suptitle(f"Kidney Stone Hospitalizations — São Paulo  (Kendall τ={tau:.3f}, p={p:.4f})",
             fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "yearly_overview", prefix="02")

print(f"Trend test: Kendall τ = {tau:.3f}, p = {p:.4f} → {'Significant' if p < 0.01 else 'Not significant'} monotonic uptrend")
print(f"CAGR (2016–2025): {cagr*100:.1f}%")
print(f"Pre-COVID mean (2016–2019): {yearly[yearly['year'].between(2016,2019)]['admissions'].mean():,.0f}/yr")
print(f"Post-COVID mean (2021–2025): {yearly[yearly['year'].between(2021,2025)]['admissions'].mean():,.0f}/yr")
print(f"2025 volume: {int(yearly[yearly['year']==2025]['admissions'].values[0]):,} (+{float(yearly[yearly['year']==2025]['yoy_growth'].values[0]):.1f}% YoY)")
print(f"Note: Dec 2025 shows 1,984 admissions vs ~2,500/mo avg — possible late DATASUS publication")

  Saved: 02_yearly_overview.png
Trend test: Kendall τ = 0.911, p = 0.0000 → Significant monotonic uptrend
CAGR (2016–2025): 9.2%
Pre-COVID mean (2016–2019): 15,839/yr
Post-COVID mean (2021–2025): 25,168/yr
2025 volume: 31,362 (+1.2% YoY)
Note: Dec 2025 shows 1,984 admissions vs ~2,500/mo avg — possible late DATASUS publication


## 2. Demographics

In [3]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Age distribution
axes[0].hist(kidney["age"], bins=50, color="#2563EB", edgecolor="white", alpha=0.8)
axes[0].axvline(kidney["age"].median(), color="red", linestyle="--",
                label=f"Median: {kidney['age'].median():.0f}")
axes[0].axvline(kidney["age"].mean(), color="orange", linestyle="--",
                label=f"Mean: {kidney['age'].mean():.1f}")
axes[0].set_title("Age Distribution")
axes[0].set_xlabel("Age")
axes[0].set_ylabel("Count")
axes[0].legend()

# Sex distribution
sex_counts = kidney["is_male"].value_counts().rename({1: "Male", 0: "Female"})
axes[1].bar(sex_counts.index, sex_counts.values, color=["#2563EB", "#DC2626"])
axes[1].set_title("Sex Distribution")
axes[1].set_ylabel("Count")
axes[1].set_xticks([0, 1])
axes[1].set_xticklabels(["Female", "Male"])
for i, v in enumerate(sex_counts.values):
    axes[1].text(i, v + 500, f"{v:,}\n({v/len(kidney)*100:.1f}%)", ha="center")

# Age trend over time
age_yr = kidney[kidney["year"] >= 2016].groupby("year")["age"].agg(["mean", "std", "count"])
age_yr["se"] = age_yr["std"] / np.sqrt(age_yr["count"])
axes[2].fill_between(age_yr.index, age_yr["mean"] - 1.96*age_yr["se"],
                     age_yr["mean"] + 1.96*age_yr["se"], alpha=0.2, color="#7C3AED")
axes[2].plot(age_yr.index, age_yr["mean"], "o-", color="#7C3AED")
axes[2].set_title("Mean Age Trend (±95% CI)")
axes[2].set_ylabel("Age (years)")
axes[2].set_xlabel("Year")

fig.tight_layout()
save_plot(fig, "demographics", prefix="02")

# Statistical tests
tau_age, p_age = stats.kendalltau(range(len(age_yr)), age_yr["mean"])
print(f"Median age: {kidney['age'].median():.0f}, IQR: [{kidney['age'].quantile(0.25):.0f}, {kidney['age'].quantile(0.75):.0f}]")
print(f"Male: {kidney['is_male'].mean()*100:.1f}%, Female: {(1-kidney['is_male'].mean())*100:.1f}%")
print(f"Age trend: Kendall τ = {tau_age:.3f}, p = {p_age:.4f} (2016 mean: {age_yr.iloc[0]['mean']:.1f} → 2024 mean: {age_yr.iloc[-2]['mean']:.1f})")

  Saved: 02_demographics.png
Median age: 46, IQR: [35, 57]
Male: 47.2%, Female: 52.8%
Age trend: Kendall τ = 0.911, p = 0.0000 (2016 mean: 44.0 → 2024 mean: 48.2)


## 3. Geographic Distribution

In [4]:
# Top treatment cities
top_cities = kidney.groupby("city_name").size().sort_values(ascending=False).head(15)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

top_cities.plot.barh(ax=axes[0], color="#2563EB")
axes[0].set_title("Top 15 Cities by Treatment Volume")
axes[0].set_xlabel("Admissions")
axes[0].invert_yaxis()

# Migration: treated outside home city
migration_rate = kidney.groupby("year")["migrated"].mean() * 100
axes[1].plot(migration_rate.index, migration_rate.values, "o-", color="#7C3AED")
axes[1].set_title("Patient Migration Rate")
axes[1].set_ylabel("% treated outside home city")
axes[1].set_xlabel("Year")

fig.tight_layout()
save_plot(fig, "geography", prefix="02")

print(f"Overall migration rate: {kidney['migrated'].mean()*100:.1f}%")

  Saved: 02_geography.png
Overall migration rate: 36.5%


## 4. Procedure Mix

In [5]:
# Category breakdown
cat_counts = kidney["proc_category"].value_counts()
cat_pct = cat_counts / len(kidney) * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cat_pct.plot.barh(ax=axes[0], color="#2563EB")
axes[0].set_title("Procedure Category Distribution")
axes[0].set_xlabel("% of admissions")
axes[0].invert_yaxis()

# Top procedures by name
proc_counts = kidney["proc_name"].value_counts().head(10)
proc_counts.plot.barh(ax=axes[1], color="#059669")
axes[1].set_title("Top 10 Procedures")
axes[1].set_xlabel("Count")
axes[1].invert_yaxis()

fig.tight_layout()
save_plot(fig, "procedures", prefix="02")

print("Category breakdown:")
for cat, pct in cat_pct.items():
    print(f"  {cat}: {pct:.1f}%")

  Saved: 02_procedures.png
Category breakdown:
  SURGICAL: 42.9%
  DIAGNOSTIC: 20.1%
  CLINICAL_MGMT: 11.3%
  SURGICAL_MGMT: 10.0%
  INTERVENTIONAL: 9.7%
  OBSERVATION: 4.3%
  OTHER: 1.7%


## 5. Hospital Classification Breakdown

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

tags["facility_type"].value_counts().plot.barh(ax=axes[0], color="#2563EB")
axes[0].set_title("Facility Type")
axes[0].invert_yaxis()

tags["admission_profile"].value_counts().plot.barh(ax=axes[1], color="#D97706")
axes[1].set_title("Admission Profile")
axes[1].invert_yaxis()

tags["case_mix_profile"].value_counts().plot.barh(ax=axes[2], color="#059669")
axes[2].set_title("Case-Mix Profile")
axes[2].invert_yaxis()

fig.suptitle("Hospital Classification (510 hospitals)", fontsize=14, fontweight="bold")
fig.tight_layout()
save_plot(fig, "hospital_classification", prefix="02")

  Saved: 02_hospital_classification.png


## 6b. Mortality with Confidence Intervals

In [7]:
mort_yr = kidney[kidney["year"] >= 2016].groupby("year")["MORTE"].agg(["sum", "mean", "count"])
mort_yr["se"] = np.sqrt(mort_yr["mean"] * (1 - mort_yr["mean"]) / mort_yr["count"])
mort_yr["ci_lo"] = (mort_yr["mean"] - 1.96 * mort_yr["se"]) * 100
mort_yr["ci_hi"] = (mort_yr["mean"] + 1.96 * mort_yr["se"]) * 100
mort_yr["rate"] = mort_yr["mean"] * 100

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].fill_between(mort_yr.index, mort_yr["ci_lo"], mort_yr["ci_hi"],
                     alpha=0.2, color="#DC2626")
axes[0].plot(mort_yr.index, mort_yr["rate"], "o-", color="#DC2626")
axes[0].set_title("Mortality Rate (±95% CI)")
axes[0].set_ylabel("Mortality (%)")
axes[0].set_xlabel("Year")
axes[0].axhline(kidney["MORTE"].mean() * 100, ls="--", color="#6B7280", alpha=0.5,
                label=f"Overall: {kidney['MORTE'].mean()*100:.3f}%")
axes[0].legend()

axes[1].bar(mort_yr.index, mort_yr["sum"], color="#DC2626", alpha=0.8)
axes[1].set_title("Absolute Deaths per Year")
axes[1].set_ylabel("Deaths")
axes[1].set_xlabel("Year")

fig.tight_layout()
save_plot(fig, "mortality_trend", prefix="02")

# Trend test
tau_m, p_m = stats.kendalltau(range(len(mort_yr)), mort_yr["rate"])
overall_se = np.sqrt(kidney["MORTE"].mean() * (1 - kidney["MORTE"].mean()) / len(kidney))
print(f"Overall mortality: {kidney['MORTE'].mean()*100:.3f}% (95% CI: [{(kidney['MORTE'].mean()-1.96*overall_se)*100:.3f}%, {(kidney['MORTE'].mean()+1.96*overall_se)*100:.3f}%])")
print(f"Mortality trend: Kendall τ = {tau_m:.3f}, p = {p_m:.4f} → {'Significant' if p_m < 0.05 else 'No significant'} trend")
print(f"Absolute deaths: {int(mort_yr.iloc[0]['sum'])} ({int(mort_yr.index[0])}) → {int(mort_yr.iloc[-2]['sum'])} ({int(mort_yr.index[-2])})")

  Saved: 02_mortality_trend.png
Overall mortality: 0.346% (95% CI: [0.320%, 0.371%])
Mortality trend: Kendall τ = -0.111, p = 0.7275 → No significant trend
Absolute deaths: 56 (2016) → 98 (2024)


## 7. Admission Type: Emergency vs Elective Shift

In [8]:
k16 = kidney[kidney["year"] >= 2016]
adm_yr = k16.groupby("year").agg(
    pct_emerg=pd.NamedAgg(column="is_emergency", aggfunc="mean"),
    los_emerg=pd.NamedAgg(column="DIAS_PERM", aggfunc=lambda x: x[k16.loc[x.index, "is_emergency"]==1].mean()),
    los_elect=pd.NamedAgg(column="DIAS_PERM", aggfunc=lambda x: x[k16.loc[x.index, "is_emergency"]==0].mean()),
)

emerg = k16[k16["is_emergency"] == 1]
elect = k16[k16["is_emergency"] == 0]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Emergency share trend
emerg_yr = k16.groupby("year")["is_emergency"].mean() * 100
axes[0].plot(emerg_yr.index, emerg_yr.values, "o-", color="#DC2626")
axes[0].set_title("Emergency Admission Share")
axes[0].set_ylabel("% Emergency")
axes[0].set_xlabel("Year")
axes[0].axhline(50, ls="--", color="#6B7280", alpha=0.5)

# LOS comparison
labels = ["Emergency", "Elective"]
means = [emerg["DIAS_PERM"].mean(), elect["DIAS_PERM"].mean()]
ci = [1.96 * emerg["DIAS_PERM"].std() / np.sqrt(len(emerg)),
      1.96 * elect["DIAS_PERM"].std() / np.sqrt(len(elect))]
axes[1].bar(labels, means, yerr=ci, color=["#DC2626", "#2563EB"], capsize=5)
axes[1].set_title("Mean LOS by Admission Type (±95% CI)")
axes[1].set_ylabel("Days")
for i, (m, c) in enumerate(zip(means, ci)):
    axes[1].text(i, m + c + 0.05, f"{m:.2f}d", ha="center", fontweight="bold")

# Mortality comparison
mort_emerg = emerg["MORTE"].mean() * 100
mort_elect = elect["MORTE"].mean() * 100
axes[2].bar(labels, [mort_emerg, mort_elect], color=["#DC2626", "#2563EB"])
axes[2].set_title("Mortality by Admission Type")
axes[2].set_ylabel("Mortality (%)")
for i, v in enumerate([mort_emerg, mort_elect]):
    axes[2].text(i, v + 0.01, f"{v:.3f}%", ha="center", fontweight="bold")

fig.tight_layout()
save_plot(fig, "admission_type", prefix="02")

# Statistical test: LOS difference
u_stat, u_p = stats.mannwhitneyu(emerg["DIAS_PERM"], elect["DIAS_PERM"], alternative="greater")
print(f"Emergency: {len(emerg):,} admissions, LOS={emerg['DIAS_PERM'].mean():.2f}d, mortality={mort_emerg:.3f}%")
print(f"Elective:  {len(elect):,} admissions, LOS={elect['DIAS_PERM'].mean():.2f}d, mortality={mort_elect:.3f}%")
print(f"LOS difference: Mann-Whitney U p = {u_p:.2e} (emergency > elective)")
print(f"Emergency share trend: {emerg_yr.iloc[0]:.1f}% (2016) → {emerg_yr.iloc[-2]:.1f}% (2024)")

  Saved: 02_admission_type.png
Emergency: 116,277 admissions, LOS=2.98d, mortality=0.513%
Elective:  89,583 admissions, LOS=1.77d, mortality=0.129%
LOS difference: Mann-Whitney U p = 0.00e+00 (emergency > elective)
Emergency share trend: 58.1% (2016) → 49.2% (2024)


## 8. Hospital & Geographic Concentration

In [9]:
hosp_vol = kidney.groupby("CNES").size().sort_values(ascending=False)
city_vol = kidney.groupby("MUNIC_MOV").size().sort_values(ascending=False)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Lorenz curve — hospital concentration
hosp_sorted = np.sort(hosp_vol.values)
hosp_cum = np.cumsum(hosp_sorted) / hosp_sorted.sum()
hosp_x = np.arange(1, len(hosp_sorted) + 1) / len(hosp_sorted)
axes[0].fill_between(hosp_x, hosp_cum, alpha=0.3, color="#2563EB")
axes[0].plot(hosp_x, hosp_cum, color="#2563EB")
axes[0].plot([0, 1], [0, 1], "--", color="#6B7280")
axes[0].set_title("Hospital Concentration (Lorenz)")
axes[0].set_xlabel("Cumulative share of hospitals")
axes[0].set_ylabel("Cumulative share of admissions")

# Hospital volume histogram
axes[1].hist(hosp_vol.values, bins=50, color="#2563EB", edgecolor="white", alpha=0.8)
axes[1].axvline(hosp_vol.median(), color="red", ls="--", label=f"Median: {hosp_vol.median():.0f}")
axes[1].set_title("Hospital Volume Distribution")
axes[1].set_xlabel("Total admissions (10yr)")
axes[1].set_ylabel("Number of hospitals")
axes[1].legend()
axes[1].set_xlim(0, 3000)

# Low-volume tail
vol_bins = pd.cut(hosp_vol, bins=[0, 10, 50, 200, 1000, hosp_vol.max()],
                  labels=["<10", "10-50", "50-200", "200-1K", ">1K"])
vol_dist = vol_bins.value_counts().sort_index()
axes[2].bar(range(len(vol_dist)), vol_dist.values, color="#7C3AED",
            tick_label=vol_dist.index)
axes[2].set_title("Hospitals by Volume Category")
axes[2].set_ylabel("Count")
for i, v in enumerate(vol_dist.values):
    axes[2].text(i, v + 2, f"{v}", ha="center", fontsize=9)

fig.tight_layout()
save_plot(fig, "concentration", prefix="02")

# Gini coefficient for hospitals
gini_hosp = 1 - 2 * hosp_cum.mean()
print(f"Hospital Gini coefficient: {gini_hosp:.3f}")
print(f"Top 10 hospitals: {hosp_vol.head(10).sum()/len(kidney)*100:.1f}% of admissions")
print(f"Top 50 hospitals: {hosp_vol.head(50).sum()/len(kidney)*100:.1f}% of admissions")
print(f"Low-volume (<10 cases in 10yr): {(hosp_vol < 10).sum()} hospitals ({(hosp_vol < 10).mean()*100:.1f}%)")
print(f"Top 10 cities: {city_vol.head(10).sum()/len(kidney)*100:.1f}% of admissions")
print(f"Migration rate: {kidney['migrated'].mean()*100:.1f}%")

  Saved: 02_concentration.png
Hospital Gini coefficient: 0.801
Top 10 hospitals: 24.0% of admissions
Top 50 hospitals: 65.7% of admissions
Low-volume (<10 cases in 10yr): 133 hospitals (26.1%)
Top 10 cities: 51.2% of admissions
Migration rate: 36.5%


## 9. LOS Distribution & Data Quality

In [10]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# LOS distribution (capped at 20 for visibility)
los_capped = kidney["DIAS_PERM"].clip(upper=20)
axes[0].hist(los_capped, bins=21, color="#2563EB", edgecolor="white", alpha=0.8)
axes[0].axvline(kidney["DIAS_PERM"].median(), color="red", linestyle="--",
                label=f"Median: {kidney['DIAS_PERM'].median():.0f}d")
axes[0].axvline(kidney["DIAS_PERM"].mean(), color="orange", linestyle="--",
                label=f"Mean: {kidney['DIAS_PERM'].mean():.1f}d")
axes[0].set_title("Length of Stay Distribution")
axes[0].set_xlabel("Days (capped at 20)")
axes[0].set_ylabel("Count")
axes[0].legend()

# Cost distribution
cost_capped = kidney["VAL_TOT"].clip(upper=5000)
axes[1].hist(cost_capped, bins=50, color="#059669", edgecolor="white", alpha=0.8)
axes[1].axvline(kidney["VAL_TOT"].median(), color="red", linestyle="--",
                label=f"Median: R${kidney['VAL_TOT'].median():,.0f}")
axes[1].set_title("Cost Distribution")
axes[1].set_xlabel("BRL (capped at 5000)")
axes[1].set_ylabel("Count")
axes[1].legend()

fig.tight_layout()
save_plot(fig, "distributions", prefix="02")

  Saved: 02_distributions.png


## Summary Metrics

In [11]:
k16_full = kidney[kidney["year"] >= 2016]
hosp_vol_m = kidney.groupby("CNES").size().sort_values(ascending=False)
hosp_sorted_m = np.sort(hosp_vol_m.values)
hosp_cum_m = np.cumsum(hosp_sorted_m) / hosp_sorted_m.sum()
overall_se_m = np.sqrt(kidney["MORTE"].mean() * (1 - kidney["MORTE"].mean()) / len(kidney))

metrics = {
    "total_admissions": int(len(kidney)),
    "total_hospitals": int(len(tags)),
    "year_range": f"{int(kidney['year'].min())}-{int(kidney['year'].max())}",
    "cagr_2016_2025": round(float(cagr * 100), 1),
    "volume_trend_kendall_tau": round(float(tau), 3),
    "volume_trend_p": float(p),
    "pre_covid_mean_yr": int(yearly[yearly["year"].between(2016, 2019)]["admissions"].mean()),
    "post_covid_mean_yr": int(yearly[yearly["year"].between(2021, 2025)]["admissions"].mean()),
    "admissions_2025": int(k16_full[k16_full["year"]==2025].shape[0]),
    "los_2025": round(float(k16_full[k16_full["year"]==2025]["DIAS_PERM"].mean()), 2),
    "median_age": float(kidney["age"].median()),
    "age_iqr": f"{int(kidney['age'].quantile(0.25))}-{int(kidney['age'].quantile(0.75))}",
    "age_trend_tau": round(float(tau_age), 3),
    "pct_male": round(float(kidney["is_male"].mean() * 100), 1),
    "pct_female": round(float((1 - kidney["is_male"].mean()) * 100), 1),
    "median_los": float(kidney["DIAS_PERM"].median()),
    "mean_los": round(float(kidney["DIAS_PERM"].mean()), 2),
    "los_2016": round(float(k16_full[k16_full["year"]==2016]["DIAS_PERM"].mean()), 2),
    "los_2024": round(float(k16_full[k16_full["year"]==2024]["DIAS_PERM"].mean()), 2),
    "pct_longstay_gt7": round(float((kidney["DIAS_PERM"] > 7).mean() * 100), 1),
    "total_cost_brl": round(float(kidney["VAL_TOT"].sum()), 2),
    "mean_cost_per_admission": round(float(kidney["VAL_TOT"].mean()), 0),
    "total_deaths": int(kidney["MORTE"].sum()),
    "mortality_rate_pct": round(float(kidney["MORTE"].mean() * 100), 3),
    "mortality_95ci_lo": round(float((kidney["MORTE"].mean() - 1.96 * overall_se_m) * 100), 3),
    "mortality_95ci_hi": round(float((kidney["MORTE"].mean() + 1.96 * overall_se_m) * 100), 3),
    "pct_emergency": round(float(kidney["is_emergency"].mean() * 100), 1),
    "emergency_2016_pct": round(float(k16_full[k16_full["year"]==2016]["is_emergency"].mean() * 100), 1),
    "emergency_2024_pct": round(float(k16_full[k16_full["year"]==2024]["is_emergency"].mean() * 100), 1),
    "migration_rate": round(float(kidney["migrated"].mean() * 100), 1),
    "top10_hospital_share": round(float(hosp_vol_m.head(10).sum() / len(kidney) * 100), 1),
    "top50_hospital_share": round(float(hosp_vol_m.head(50).sum() / len(kidney) * 100), 1),
    "hospital_gini": round(float(1 - 2 * hosp_cum_m.mean()), 3),
    "low_volume_hospitals_lt10": int((hosp_vol_m < 10).sum()),
    "comparability_groups": int(tags["comparability_group"].nunique()),
    "pct_diagnostic_admissions": round(float((kidney["proc_category"]=="DIAGNOSTIC").mean() * 100), 1),
}
save_metrics(metrics, "02_general_overview")

print("\n=== OVERVIEW METRICS ===")
for k, v in metrics.items():
    print(f"  {k}: {v}")

  Saved metrics: 02_general_overview.json

=== OVERVIEW METRICS ===
  total_admissions: 206500
  total_hospitals: 510
  year_range: 2015-2025
  cagr_2016_2025: 9.2
  volume_trend_kendall_tau: 0.911
  volume_trend_p: 2.9761904761904762e-05
  pre_covid_mean_yr: 15838
  post_covid_mean_yr: 25168
  admissions_2025: 31362
  los_2025: 2.13
  median_age: 46.0
  age_iqr: 35-57
  age_trend_tau: 0.911
  pct_male: 47.2
  pct_female: 52.8
  median_los: 2.0
  mean_los: 2.46
  los_2016: 2.83
  los_2024: 2.17
  pct_longstay_gt7: 4.5
  total_cost_brl: 187823155.55
  mean_cost_per_admission: 910.0
  total_deaths: 714
  mortality_rate_pct: 0.346
  mortality_95ci_lo: 0.32
  mortality_95ci_hi: 0.371
  pct_emergency: 56.5
  emergency_2016_pct: 58.1
  emergency_2024_pct: 49.2
  migration_rate: 36.5
  top10_hospital_share: 24.0
  top50_hospital_share: 65.7
  hospital_gini: 0.801
  low_volume_hospitals_lt10: 133
  comparability_groups: 31
  pct_diagnostic_admissions: 20.1
